# Algoritmo escolhido: KNN
Como funciona:
- Armazena todo o conjunto de treino.
- Para classificar um novo ponto, calcula-se a distância (ex: Euclidiana) para todos os   pontos de treino.
- Seleciona os k vizinhos mais próximos.
- A classe é decidida por votação maioritária.

Razões por trás da escolha deste algoritmo:
- Simplicidade de implementação e interpretação.
- Facilmente modificável (pesos, kernels, remoção de pontos).
- Sensível ao desbalanceamento 


Hipóteses:
k-NN padrão será fortemente afetado pelo desequilíbrio. Como a classificação é feita por votação maioritária dos k vizinhos mais próximos, em datasets desequilibrados, a probabilidade de um ponto da classe minoritária ter vizinhos da classe majoritária é elevada, levando a um elevado número de Falsos Negativos e um baixo Recall.

# Função de knn normal

In [21]:
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler

class KNearstNeighbor:
    
    def __init__(self):
        # Inicializar as variáveis para podermos aceder a elas em qualquer parte da classe
        self.x_train = None
        self.y_train = None
        self.x_test = None
        self.y_test = None

    def load_split_train(self, X, y, test_size=0.2, random_state=42):
        """
        Recebe o dataset completo, faz o split de treino/teste, garante a estratificação 
        para dados desequilibrados e guarda os dados de treino (faz o fit).
        Retorna os dados de teste para poderes usar na avaliação.
        """
        # 1. Fazer o split
        self.x_train, self.x_test, self.y_train, self.y_test = train_test_split(
            X, y, 
            test_size=test_size, 
            random_state=random_state, 
            stratify=y # Mantém a proporção da classe minoritária
        )
        
        # O KNN não tem matemática de treino, apenas guarda os dados na memória.
        # Como já guardámos o x_train e y_train na linha acima, o "fit" está feito.
        
        return self.x_test, self.y_test

    # Método train original
    def train(self, x_train, y_train):
        self.x_train = x_train
        self.y_train = y_train
    
    def predict(self, x_data, k=5):
        nearst_neighbors = []
        for x in x_data:
            distances_between_points = self.__euclidean_distance(
                point_a=x,
                points_b=self.x_train,
                labels_b=self.y_train)

            # sort by distance and carry label within
            sorted_distances = sorted(
                distances_between_points, key=lambda t: t[0])
            nearst_neighbors.append(
                sorted_distances[:k]
            )

        return self.__vote(nearst_neighbors)

    def __vote(self, nearst_neighbors):
        predicts = []
        for neighbors in nearst_neighbors:
            label_counter = Counter([neighbor[1] for neighbor in neighbors])
            [(predicted_label, _)] = label_counter.most_common(1)
            predicts.append([predicted_label])
        return predicts

    def __euclidean_distance(self, point_a, points_b, labels_b):
        distances = []
        size_points_b = len(points_b)
        for i in range(size_points_b):
            point = points_b[i]
            #garante que lê bem a label seja array 1D ou 2D
            point_label = labels_b[i][0] if isinstance(labels_b[i], (list, np.ndarray)) else labels_b[i]

            distance = np.sqrt(np.sum(np.square(
                    point_a - point)))
            distances.append(
                (distance, point_label))
        return distances

    @staticmethod # Transformado em staticmethod para o poderes chamar facilmente sem precisar do 'self'
    def avaliar_desequilibrio(y_real, y_pred, classe_minoritaria):
        y_real = np.array(y_real).flatten()
        y_pred = np.array(y_pred).flatten()
        
        # Verdadeiros Positivos, Falsos Positivos, etc.
        TP = np.sum((y_pred == classe_minoritaria) & (y_real == classe_minoritaria))
        FP = np.sum((y_pred == classe_minoritaria) & (y_real != classe_minoritaria))
        FN = np.sum((y_pred != classe_minoritaria) & (y_real == classe_minoritaria))
        
        # Calcular métricas (com proteção contra divisão por zero)
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        accuracy = np.mean(y_pred == y_real)
        
        print(f"Accuracy:  {accuracy:.4f}")#(Enganadora em dados desequilibrados)
        print(f"Precision: {precision:.4f}")#(Quando diz que é anomalia, qual a % de acerto?)
        print(f"Recall:    {recall:.4f}")#(Das anomalias totais, quantas conseguiu apanhar?)
        print(f"F1-Score:  {f1_score:.4f}")#(O equilíbrio entre Precision e Recall)
        
        return accuracy, precision, recall, f1_score

In [22]:
import os
import pandas as pd
import numpy as np
from collections import Counter

# Pasta com os ficheiros CSV
pasta_datasets = 'class_imbalance'

# Instanciar o modelo
knn_normal = KNearstNeighbor()

# Iniciar o ciclo for para percorrer a pasta com os datasets
for ficheiro in os.listdir(pasta_datasets):
    
    # Garantir que só lê ficheiros de dados (ex: .csv)
    if ficheiro.endswith('.csv'):
        print("-" * 40 + "\n")
        print(f" A testar o dataset: {ficheiro}")
        
        caminho_completo = os.path.join(pasta_datasets, ficheiro)
        
        
        df = pd.read_csv(caminho_completo)
        
        # Separar features (X) da target (y)
        X_df = df.iloc[:, :-1]
        y_completo = df.iloc[:, -1].values
        
        # Transformar todas as colunas de TEXTO em NÚMEROS (One-Hot Encoding)
        # O drop_first=True evita colinearidade matemática perfeita
        X_df = pd.get_dummies(X_df, drop_first=True)
        
        # Preencher possíveis valores vazios (NaN) com 0 
        X_df = X_df.fillna(0)
        
        # Converter para array do NumPy e garantir que é tudo número (float)
        X_completo = X_df.values.astype(float)

        
        # DETETAR A CLASSE MINORITÁRIA AUTOMATICAMENTE
        contagem = Counter(y_completo)
        classe_minoritaria = min(contagem, key=contagem.get)
        
        print(f"Distribuição total do dataset: {dict(contagem)}")
        print(f"A Classe Minoritária é: '{classe_minoritaria}'\n")
        
        # SPLIT + FIT
        x_test, y_test = knn_normal.load_split_train(X_completo, y_completo, test_size=0.3)

        # NORMALIZAÇÃO
        scaler = StandardScaler()
        # O scaler APRENDE no treino
        knn_normal.x_train = scaler.fit_transform(knn_normal.x_train)
        # O scaler apenas APLICA no teste (para evitar leakage)
        x_test_scaled = scaler.transform(x_test)
        
        # PREVER
        previsoes = knn_normal.predict(x_test_scaled, k=5)
        
        # AVALIAR
        print("--- Resultados KNN ---")
        KNearstNeighbor.avaliar_desequilibrio(y_test, previsoes, classe_minoritaria=classe_minoritaria)
        print("\n\n")

----------------------------------------

 A testar o dataset: dataset_978_mfeat-factors.csv
Distribuição total do dataset: {'P': 200, 'N': 1800}
A Classe Minoritária é: 'P'

--- Resultados KNN ---
Accuracy:  0.9967
Precision: 0.9677
Recall:    1.0000
F1-Score:  0.9836



----------------------------------------

 A testar o dataset: dataset_947_arsenic-male-bladder.csv
Distribuição total do dataset: {'N': 24, 'P': 535}
A Classe Minoritária é: 'N'

--- Resultados KNN ---
Accuracy:  0.9762
Precision: 1.0000
Recall:    0.4286
F1-Score:  0.6000



----------------------------------------

 A testar o dataset: dataset_1004_synthetic_control.csv
Distribuição total do dataset: {'N': 500, 'P': 100}
A Classe Minoritária é: 'P'

--- Resultados KNN ---
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1-Score:  1.0000



----------------------------------------

 A testar o dataset: dataset_1056_mc1.csv
Distribuição total do dataset: {np.False_: 9398, np.True_: 68}
A Classe Minoritária é: '

# Algoritmo CW_RBF_KNN mas sem Sequential Insertion
Nota: usa cross fold validation

In [12]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

class CW_RBF_KNN:
    def __init__(self):
        self.gamma = None
        self.class_weights = {}
        self.feature_weights = []
        self.x_train = None
        self.y_train = None

    def load_split_train(self, X, y, test_size=0.2, random_state=42):
        """
        Faz o split e carrega os dados diretamente para o treino, 
        sem qualquer processo de seleção/eliminação de pontos.
        """
        x_train_raw, x_test, y_train_raw, y_test = train_test_split(
            X, y, 
            test_size=test_size, 
            random_state=random_state, 
            stratify=y 
        )
        
        # Chama o train simplificado
        self.train(x_train_raw, y_train_raw)
        
        return x_test, y_test

    def train(self, x_raw, y_raw):
        """
        Versão SEM Sequential Insertion. 
        Guarda todos os dados e calcula os pesos globais.
        """
        # Converter para numpy arrays para garantir compatibilidade
        self.x_train = np.array(x_raw)
        self.y_train = np.array(y_raw).flatten()
        
        # Calcula os pesos (Gamma, Class Weights e Feature Weights) uma única vez
        # para o conjunto total de treino.
        self.__update_internal_weights(self.x_train, self.y_train)

    def __update_internal_weights(self, x_data, y_data):
        """Calcula pesos baseados na totalidade dos dados de treino."""
        total_samples = len(y_data)
        counts = Counter(y_data)

        # 1. Cálculo do Gamma (RBF)
        if len(x_data) > 1:
            sample = x_data[:min(200, len(x_data))]
            dists = []
            for i in range(len(sample)):
                for j in range(i+1, len(sample)):
                    dists.append(np.sum(np.square(sample[i] - sample[j])))
            median_dist = np.median(dists)
            self.gamma = 1.0 / (2 * median_dist) if median_dist > 0 else 1.0
        else:
            self.gamma = 1.0 / x_data.shape[1]

        # 2. Pesos das Classes
        self.class_weights = {label: np.sqrt(total_samples / count) for label, count in counts.items()}

        # 3. Pesos das Features (Correlação)
        if len(counts) > 1:
            unique_labels = list(counts.keys())
            # Binarização simples para correlação
            y_numeric = np.where(y_data == unique_labels[0], 0, 1)
            
            f_weights = []
            for i in range(x_data.shape[1]):
                coluna = x_data[:, i]
                if np.std(coluna) == 0:
                    f_weights.append(0.0)
                else:
                    corr = np.corrcoef(coluna, y_numeric)[0, 1]
                    f_weights.append(np.abs(corr) if not np.isnan(corr) else 0.0)
            self.feature_weights = np.array(f_weights)
        else:
            self.feature_weights = np.ones(x_data.shape[1])

    def predict(self, x_data, k=5):
        x_data = np.array(x_data)
        predicts = []
        for x in x_data:
            similarities = self.__rbf_similarity(x)
            # No RBF, maior similaridade significa mais próximo
            sorted_similarities = sorted(similarities, key=lambda t: t[0], reverse=True)
            k_nearest = sorted_similarities[:k]
            predicts.append(self.__vote(k_nearest))
        return np.array(predicts)

    def __rbf_similarity(self, point_a):
        diferenca_quadrado = np.square(self.x_train - point_a)
        # Distância Euclidiana Ponderada pelas features
        distancia_ponderada = np.sum(self.feature_weights * diferenca_quadrado, axis=1)
        # Kernel RBF
        similarities = np.exp(-self.gamma * distancia_ponderada)
        return list(zip(similarities, self.y_train))

    def __vote(self, neighbors):
        class_scores = {}
        for similarity, label in neighbors:
            # Voto ponderado pela similaridade RBF e pelo peso da classe
            vote_power = self.class_weights.get(label, 1.0) * similarity
            class_scores[label] = class_scores.get(label, 0) + vote_power
        
        return max(class_scores, key=class_scores.get)

    def evaluate(self, x_test, y_test, k=5):
        y_real = np.array(y_test).flatten()
        y_pred = self.predict(x_test, k=k)
        
        counts = Counter(y_real)
        minority = min(counts, key=counts.get)
        
        tp = np.sum((y_pred == minority) & (y_real == minority))
        fp = np.sum((y_pred == minority) & (y_real != minority))
        fn = np.sum((y_pred != minority) & (y_real == minority))
        
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0
        acc = np.mean(y_pred == y_real)
        
        return acc, prec, rec, f1

In [15]:
import os
from sklearn.model_selection import StratifiedKFold
import numpy as np

# Pasta com os ficheiros CSV
pasta_datasets = 'class_imbalance'

# Cabeçalho da Tabela para a avaliação empírica
header = f"{'Dataset':<35} | {'Orig':>6} | {'Sel':>6} | {'Acc':>7} | {'Prec':>6} | {'Rec':>6} | {'F1':>6}"
print(header)
print("-" * len(header))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) #k fold cross validation estratificado e definimos 5 folds (n_splits)
#(o estratificado garante que a proporção é mais ou menos igual tanto no teste como no treino)

for ficheiro in os.listdir(pasta_datasets):
    if ficheiro.endswith('.csv'):
        try:
            caminho_completo = os.path.join(pasta_datasets, ficheiro)
            df = pd.read_csv(caminho_completo)
            
            # Separar features (X) da target (y)
            X_df = df.iloc[:, :-1]
            y_completo = df.iloc[:, -1].values
            
            # Transformar todas as colunas de TEXTO em NÚMEROS (One-Hot Encoding)
            # O drop_first=True evita colinearidade matemática perfeita
            X_df = pd.get_dummies(X_df, drop_first=True)
            
            # Preencher possíveis valores vazios (NaN) com 0 
            X_df = X_df.fillna(0)
            
            # Converter para array do NumPy e garantir que é tudo número (float)
            X_completo = X_df.values.astype(float)
            
            # Calcular tamanho do treino (antes da compressão) para a tabela
            orig_train_size = int(len(y_completo) * 0.7) 

            # Listas para guardar as métricas de cada fold
            fold_acc, fold_prec, fold_rec, fold_f1, fold_sel_size = [], [], [], [], []
            
            for train_index, test_index in skf.split(X_completo, y_completo):
                # Dividir os dados
                X_train_cv, X_test_cv = X_completo[train_index], X_completo[test_index]
                y_train_cv, y_test_cv = y_completo[train_index], y_completo[test_index]

                # NORMALIZAÇÃO
                scaler = StandardScaler()
                # O scaler APRENDE no treino
                X_train_cv = scaler.fit_transform(X_train_cv)
                # O scaler apenas APLICA no teste (para evitar leakage)
                X_test_cv = scaler.transform(X_test_cv)

                # Instanciar e treinar
                model = CW_RBF_KNN()
                model.train(X_train_cv, y_train_cv)

                # Avaliar o fold atual
                acc, prec, rec, f1 = model.evaluate(X_test_cv, y_test_cv, k=5)

                #adicionamos aos arrays de resultados
                fold_acc.append(acc)
                fold_prec.append(prec)
                fold_rec.append(rec)
                fold_f1.append(f1)
                fold_sel_size.append(len(model.x_train)) # Guarda quantos pontos o CNN selecionou neste fold
            
            # tamanho do treino
            orig_size = len(y_completo)
            avg_sel_size = np.mean(fold_sel_size)
            
            # Imprimir linha formatada (Os resultados dão as médias dos arrays obtidos anteriomente)
            print(f"{ficheiro[:35]:<35} | {orig_train_size:>6} | {avg_sel_size:>6} | {np.mean(fold_acc)*100:>6.1f}% | {np.mean(fold_prec)*100:>6.2f} | {np.mean(fold_rec)*100:>6.2f} | {np.mean(fold_f1)*100:>6.2f}")
            
        except Exception as e:
            print(f"Erro no dataset {ficheiro}: {e}")

print("-" * len(header))
print("Avaliação empírica concluída.")

Dataset                             |   Orig |    Sel |     Acc |   Prec |    Rec |     F1
------------------------------------------------------------------------------------------
dataset_978_mfeat-factors.csv       |   1400 | 1600.0 |   99.4% |  95.67 |  98.00 |  96.80
dataset_947_arsenic-male-bladder.cs |    391 |  447.2 |   86.9% |  17.97 |  58.00 |  27.33
dataset_1004_synthetic_control.csv  |    420 |  480.0 |  100.0% | 100.00 | 100.00 | 100.00
dataset_1056_mc1.csv                |   6626 | 7572.8 |   98.3% |  25.63 |  72.20 |  37.74
dataset_940_water-treatment.csv     |    368 |  421.6 |   80.1% |  35.61 |  33.75 |  33.35
dataset_950_arsenic-female-lung.csv |    391 |  447.2 |   95.3% |  42.83 |  78.33 |  54.31
dataset_1014_analcatdata_dmft.csv   |    557 |  637.6 |   71.1% |  15.43 |  10.97 |  12.61
dataset_1039_hiva_agnostic.csv      |   2960 | 3383.2 |   90.8% |  18.95 |  48.97 |  27.30
dataset_1018_ipums_la_99-small.csv  |   6190 | 7075.2 |   90.5% |  25.53 |  24.47 |  24.94

# Algoritmo CW_RBF_KNN mas sem o RBF:
Nota: este também não tem a normalizaçaão

In [16]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

class CW_KNN_Sequential:
    def __init__(self):
        # Removido o Gamma (RBF)
        self.class_weights = {}
        self.feature_weights = []
        self.x_train = None
        self.y_train = None

    def load_split_train(self, X, y, test_size=0.2, random_state=42):
        x_train_raw, x_test, y_train_raw, y_test = train_test_split(
            X, y, 
            test_size=test_size, 
            random_state=random_state, 
            stratify=y
        )
        
        # Executa o treino com a lógica de Sequential Insertion
        self.train(x_train_raw, y_train_raw)
        
        return x_test, y_test

    def train(self, x_raw, y_raw):
        x_raw = np.array(x_raw)
        y_raw = np.array(y_raw).flatten()
        
        self.__update_internal_weights(x_raw, y_raw)
        
        counts = Counter(y_raw)
        minority_class = min(counts, key=counts.get)
        
        # Início da seleção: Todos os pontos da classe minoritária entram
        minority_mask = y_raw == minority_class
        self.x_train = x_raw[minority_mask]
        self.y_train = y_raw[minority_mask]
        
        # Garante pelo menos um exemplo de cada uma das outras classes
        for label in np.unique(y_raw):
            if label != minority_class:
                idx = np.where(y_raw == label)[0][0]
                self.x_train = np.vstack([self.x_train, x_raw[idx]])
                self.y_train = np.append(self.y_train, label)
        
        # Lógica de Sequential Insertion (CNN)
        changed = True
        max_loops = 3
        loops = 0
        while changed and loops < max_loops:
            changed = False
            loops += 1
            for i in range(len(x_raw)):
                if y_raw[i] == minority_class:
                    continue 
                
                # Se o modelo atual errar a classificação do ponto, inserimos o ponto no treino
                pred = self.predict([x_raw[i]], k=5)[0]
                if pred != y_raw[i]:
                    self.x_train = np.vstack([self.x_train, x_raw[i]])
                    self.y_train = np.append(self.y_train, y_raw[i])
                    changed = True
            
            if changed:
                self.__update_internal_weights(self.x_train, self.y_train)

    def __update_internal_weights(self, x_data, y_data):
        total_samples = len(y_data)
        counts = Counter(y_data)

        # PESOS DAS CLASSES (Raiz quadrada para suavizar o desequilíbrio)
        self.class_weights = {label: np.sqrt(total_samples / count) for label, count in counts.items()}

        # PESOS DAS VARIÁVEIS (Correlação Absoluta com o Target)
        if len(counts) > 1:
            unique_labels = list(counts.keys())
            y_numeric = np.where(y_data == unique_labels[0], 0, 1)
            
            f_weights = []
            for i in range(x_data.shape[1]):
                coluna = x_data[:, i]
                if np.std(coluna) == 0:
                    f_weights.append(0.0)
                else:
                    corr = np.corrcoef(coluna, y_numeric)[0, 1]
                    f_weights.append(np.abs(corr) if not np.isnan(corr) else 1.0)
            self.feature_weights = np.array(f_weights)
        else:
            self.feature_weights = np.ones(x_data.shape[1])

    def predict(self, x_data, k=5):
        x_data = np.array(x_data)
        predicts = []
        for x in x_data:
            # Substituído RBF por Distância Euclidiana Ponderada
            distances = self.__weighted_euclidean_distance(x)
            
            # Ordenar por distância CRESCENTE (menor distância = mais próximo)
            sorted_distances = sorted(distances, key=lambda t: t[0])
            k_nearest = sorted_distances[:k]
            predicts.append(self.__vote(k_nearest))
        return np.array(predicts)

    def __weighted_euclidean_distance(self, point_a):
        # Cálculo da distância euclidiana ponderada pelos feature_weights
        diferenca_quadrado = np.square(self.x_train - point_a)
        # Multiplicamos a importância de cada coluna antes da soma
        distancia_ponderada = np.sqrt(np.sum(self.feature_weights * diferenca_quadrado, axis=1))
        return list(zip(distancia_ponderada, self.y_train))

    def __vote(self, neighbors):
        class_scores = {}
        for dist, label in neighbors:
            # Para o voto, usamos o inverso da distância (evita divisão por zero)
            # Assim, pontos mais próximos têm mais peso.
            similarity_weight = 1.0 / (dist + 1e-6)
            
            # Peso final do voto = Importância da Classe * Proximidade
            vote_power = self.class_weights.get(label, 1.0) * similarity_weight
            class_scores[label] = class_scores.get(label, 0) + vote_power
        
        return max(class_scores, key=class_scores.get)

    def evaluate(self, x_test, y_test, k=5):
        y_real = np.array(y_test).flatten()
        y_pred = self.predict(x_test, k=k)
        
        counts = Counter(y_real)
        minority = min(counts, key=counts.get)
        
        tp = np.sum((y_pred == minority) & (y_real == minority))
        fp = np.sum((y_pred == minority) & (y_real != minority))
        fn = np.sum((y_pred != minority) & (y_real == minority))
        
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0
        acc = np.mean(y_pred == y_real)
        
        return acc, prec, rec, f1

In [17]:
import os
from sklearn.model_selection import StratifiedKFold
import numpy as np

# Pasta com os ficheiros CSV
pasta_datasets = 'class_imbalance'

# Cabeçalho da Tabela para a avaliação empírica
header = f"{'Dataset':<35} | {'Orig':>6} | {'Sel':>6} | {'Acc':>7} | {'Prec':>6} | {'Rec':>6} | {'F1':>6}"
print(header)
print("-" * len(header))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) #k fold cross validation estratificado e definimos 5 folds (n_splits)
#(o estratificado garante que a proporção é mais ou menos igual tanto no teste como no treino)

for ficheiro in os.listdir(pasta_datasets):
    if ficheiro.endswith('.csv'):
        try:
            caminho_completo = os.path.join(pasta_datasets, ficheiro)
            df = pd.read_csv(caminho_completo)
            
            # Separar features (X) da target (y)
            X_df = df.iloc[:, :-1]
            y_completo = df.iloc[:, -1].values
            
            # Transformar todas as colunas de TEXTO em NÚMEROS (One-Hot Encoding)
            # O drop_first=True evita colinearidade matemática perfeita
            X_df = pd.get_dummies(X_df, drop_first=True)
            
            # Preencher possíveis valores vazios (NaN) com 0 
            X_df = X_df.fillna(0)
            
            # Converter para array do NumPy e garantir que é tudo número (float)
            X_completo = X_df.values.astype(float)
            
            # Calcular tamanho do treino (antes da compressão) para a tabela
            orig_train_size = int(len(y_completo) * 0.7) 

            # Listas para guardar as métricas de cada fold
            fold_acc, fold_prec, fold_rec, fold_f1, fold_sel_size = [], [], [], [], []
            
            for train_index, test_index in skf.split(X_completo, y_completo):
                # Dividir os dados
                X_train_cv, X_test_cv = X_completo[train_index], X_completo[test_index]
                y_train_cv, y_test_cv = y_completo[train_index], y_completo[test_index]

                # NORMALIZAÇÃO
                scaler = StandardScaler()
                # O scaler APRENDE no treino
                X_train_cv = scaler.fit_transform(X_train_cv)
                # O scaler apenas APLICA no teste (para evitar leakage)
                X_test_cv = scaler.transform(X_test_cv)


                # Instanciar e treinar
                model = CW_RBF_KNN()
                model.train(X_train_cv, y_train_cv)

                # Avaliar o fold atual
                acc, prec, rec, f1 = model.evaluate(X_test_cv, y_test_cv, k=5)

                #adicionamos aos arrays de resultados
                fold_acc.append(acc)
                fold_prec.append(prec)
                fold_rec.append(rec)
                fold_f1.append(f1)
                fold_sel_size.append(len(model.x_train)) # Guarda quantos pontos o CNN selecionou neste fold
            
            # tamanho do treino
            orig_size = len(y_completo)
            avg_sel_size = np.mean(fold_sel_size)
            
            # Imprimir linha formatada (Os resultados dão as médias dos arrays obtidos anteriomente)
            print(f"{ficheiro[:35]:<35} | {orig_train_size:>6} | {avg_sel_size:>6} | {np.mean(fold_acc)*100:>6.1f}% | {np.mean(fold_prec):>6.2f} | {np.mean(fold_rec):>6.2f} | {np.mean(fold_f1):>6.2f}")
            
        except Exception as e:
            print(f"Erro no dataset {ficheiro}: {e}")

print("-" * len(header))
print("Avaliação empírica concluída.")

Dataset                             |   Orig |    Sel |     Acc |   Prec |    Rec |     F1
------------------------------------------------------------------------------------------
dataset_978_mfeat-factors.csv       |   1400 | 1600.0 |   99.4% |   0.96 |   0.98 |   0.97
dataset_947_arsenic-male-bladder.cs |    391 |  447.2 |   86.9% |   0.18 |   0.58 |   0.27
dataset_1004_synthetic_control.csv  |    420 |  480.0 |  100.0% |   1.00 |   1.00 |   1.00
dataset_1056_mc1.csv                |   6626 | 7572.8 |   98.3% |   0.26 |   0.72 |   0.38
dataset_940_water-treatment.csv     |    368 |  421.6 |   80.1% |   0.36 |   0.34 |   0.33
dataset_950_arsenic-female-lung.csv |    391 |  447.2 |   95.3% |   0.43 |   0.78 |   0.54
dataset_1014_analcatdata_dmft.csv   |    557 |  637.6 |   71.1% |   0.15 |   0.11 |   0.13
dataset_1039_hiva_agnostic.csv      |   2960 | 3383.2 |   90.8% |   0.19 |   0.49 |   0.27
dataset_1018_ipums_la_99-small.csv  |   6190 | 7075.2 |   90.5% |   0.26 |   0.24 |   0.25

# Algoritmo CW_RBF_KNN:
- atribuímos maior peso a classes e atributos
  (O __update_internal_weights calcula a correlação entre cada característica e classe alvo senod que características que tÊm pouca correlação com a classe alvo recebem pesos menores)
- RBF Kernel (o método __rbf_similarity transforma a distÂncia numa pontuação de similaridade entre 0 e 1 sendo que o parâmetro gama controla o raio de influÊncia - vizinhos próximos ganham muito mais importância)
- implementámos o sequential selection, começando com um ponto de cada classe para não não demorar muito a encontrar o primeiro erro caso o primeiro nó fosse da classe maioritária


!Se supostamente não podemos fazer a normalização, tirei-a, resolvendo também o prolema de data leakage


In [18]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

class CW_RBF_KNN:
    def __init__(self):
        # Hiperparâmetro que controla a largura da "bolha" de influência do Kernel RBF
        self.gamma = None
        self.class_weights = {}
        self.feature_weights = []
        self.x_train = None
        self.y_train = None

    def load_split_train(self, X, y, test_size=0.2, random_state=42):
        """
        Recebe o dataset completo, faz o split de treino/teste, garante a estratificação 
        para dados desequilibrados e inicia o processo de Insertion Selection.
        """
        x_train_raw, x_test, y_train_raw, y_test = train_test_split(
            X, y, 
            test_size=test_size, 
            random_state=random_state, 
            stratify=y # Mantém a proporção da classe minoritária
        )
        
        # O método train aqui executa a Sequential Insertion Selection (CNN)
        self.train(x_train_raw, y_train_raw)
        
        return x_test, y_test

    def train(self, x_raw, y_raw):
        x_raw = np.array(x_raw)
        y_raw = np.array(y_raw).flatten()
        
        self.__update_internal_weights(x_raw, y_raw)
        
        counts = Counter(y_raw)
        minority_class = min(counts, key=counts.get)
        
        # Começa com TODOS os pontos da classe minoritária
        minority_mask = y_raw == minority_class
        self.x_train = x_raw[minority_mask]
        self.y_train = y_raw[minority_mask]
        
        # Adiciona apenas 1 ponto de cada classe maioritária para ter referência
        for label in np.unique(y_raw):
            if label != minority_class:
                idx = np.where(y_raw == label)[0][0]
                self.x_train = np.vstack([self.x_train, x_raw[idx]])
                self.y_train = np.append(self.y_train, label)
        
        # Depois aplica o sequential insertion selection apenas nos pontos maioritários        
        changed = True
        max_loops = 3
        loops = 0
        while changed and loops < max_loops:
            changed = False
            loops += 1
            for i in range(len(x_raw)):
                if y_raw[i] == minority_class:
                    continue  # já estão todos dentro
                pred = self.predict([x_raw[i]], k=5)[0]
                if pred != y_raw[i]:
                    self.x_train = np.vstack([self.x_train, x_raw[i]])
                    self.y_train = np.append(self.y_train, y_raw[i])
                    changed = True
            if changed:
                self.__update_internal_weights(self.x_train, self.y_train)

    def __update_internal_weights(self, x_data, y_data):
        """Função auxiliar para atualizar pesos de classes e atributos durante a inserção"""
        total_samples = len(y_data)
        counts = Counter(y_data)

        # Gamma pela mediana das distâncias (muito mais estável)
        if len(x_data) > 1:
            # Amostra para não ser lento
            sample = x_data[:min(200, len(x_data))]
            dists = []
            for i in range(len(sample)):
                for j in range(i+1, len(sample)):
                    dists.append(np.sum(np.square(sample[i] - sample[j])))
            median_dist = np.median(dists)
            self.gamma = 1.0 / (2 * median_dist) if median_dist > 0 else 1.0
        else:
            self.gamma = 1.0 / x_data.shape[1]

        # PESOS DAS CLASSES (Suavizados com Raiz Quadrada)
        # Dá mais poder de voto à classe minoritária
        #alpha = 0.3 #na eventualidade de estarmos a dar demasiado peso à classe minoritária
        self.class_weights = {label: np.sqrt(total_samples / count) for label, count in counts.items()}

        # PESOS DAS VARIÁVEIS (Correlação Absoluta)
        if len(counts) > 1:
            unique_labels = list(counts.keys())
            y_numeric = np.where(y_data == unique_labels[0], 0, 1)
            
            f_weights = []
            for i in range(x_data.shape[1]):
                coluna = x_data[:, i]
                if np.std(coluna) == 0:
                    f_weights.append(0.0)
                else:
                    corr = np.corrcoef(coluna, y_numeric)[0, 1]
                    f_weights.append(np.abs(corr) if not np.isnan(corr) else 0.0)
            self.feature_weights = np.array(f_weights)
        else:
            self.feature_weights = np.ones(x_data.shape[1])

    def predict(self, x_data, k=5):
        x_data = np.array(x_data)
        predicts = []
        for x in x_data:
            # Calcular a Similaridade RBF em relação a todos os pontos de treino selecionados
            similarities = self.__rbf_similarity(x)
            # Ordenar de forma DECRESCENTE (no RBF, maior = melhor/mais perto)
            sorted_similarities = sorted(similarities, key=lambda t: t[0], reverse=True)
            k_nearest = sorted_similarities[:k]
            predicts.append(self.__vote(k_nearest))
        return np.array(predicts)

    def __rbf_similarity(self, point_a):
        # Diferença ponderada pelos feature_weights (Correlação)
        diferenca_quadrado = np.square(self.x_train - point_a)
        distancia_ponderada = np.sum(self.feature_weights * diferenca_quadrado, axis=1)
        # Aplicar o Kernel RBF
        similarities = np.exp(-self.gamma * distancia_ponderada)
        return list(zip(similarities, self.y_train))

    def __vote(self, neighbors):
        class_scores = {}
        for similarity, label in neighbors:
            vote_power = self.class_weights.get(label, 1.0) * similarity
            class_scores[label] = class_scores.get(label, 0) + vote_power
        
        """counts = Counter(self.y_train)
        minority = min(counts, key=counts.get)
        
        # Se a classe minoritária tem pelo menos 40% do score da maioritária, ganha
        minority_score = class_scores.get(minority, 0)
        max_score = max(class_scores.values())
        
         if minority_score >= 0.4 * max_score:
            return minority """
        return max(class_scores, key=class_scores.get)

    def evaluate(self, x_test, y_test, k=5):
        """Calcula métricas focadas em desequilíbrio para a classe minoritária."""
        y_real = np.array(y_test).flatten()
        y_pred = self.predict(x_test, k=k)
        
        counts = Counter(y_real)
        minority = min(counts, key=counts.get)
        
        tp = np.sum((y_pred == minority) & (y_real == minority))
        fp = np.sum((y_pred == minority) & (y_real != minority))
        fn = np.sum((y_pred != minority) & (y_real == minority))
        
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0
        acc = np.mean(y_pred == y_real)
        
        return acc, prec, rec, f1

In [19]:
import os
from sklearn.model_selection import StratifiedKFold
import numpy as np

# Pasta com os ficheiros CSV
pasta_datasets = 'class_imbalance'

# Cabeçalho da Tabela para a avaliação empírica
header = f"{'Dataset':<35} | {'Orig':>6} | {'Sel':>6} | {'Acc':>7} | {'Prec':>6} | {'Rec':>6} | {'F1':>6}"
print(header)
print("-" * len(header))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) #k fold cross validation estratificado e definimos 5 folds (n_splits)
#(o estratificado garante que a proporção é mais ou menos igual tanto no teste como no treino)

for ficheiro in os.listdir(pasta_datasets):
    if ficheiro.endswith('.csv'):
        try:
            caminho_completo = os.path.join(pasta_datasets, ficheiro)
            df = pd.read_csv(caminho_completo)
            
            # Separar features (X) da target (y)
            X_df = df.iloc[:, :-1]
            y_completo = df.iloc[:, -1].values
            
            # Transformar todas as colunas de TEXTO em NÚMEROS (One-Hot Encoding)
            # O drop_first=True evita colinearidade matemática perfeita
            X_df = pd.get_dummies(X_df, drop_first=True)
            
            # Preencher possíveis valores vazios (NaN) com 0 
            X_df = X_df.fillna(0)
            
            # Converter para array do NumPy e garantir que é tudo número (float)
            X_completo = X_df.values.astype(float)
            
            # Calcular tamanho do treino (antes da compressão) para a tabela
            orig_train_size = int(len(y_completo) * 0.7) 

            # Listas para guardar as métricas de cada fold
            fold_acc, fold_prec, fold_rec, fold_f1, fold_sel_size = [], [], [], [], []
            
            for train_index, test_index in skf.split(X_completo, y_completo):
                # Dividir os dados
                X_train_cv, X_test_cv = X_completo[train_index], X_completo[test_index]
                y_train_cv, y_test_cv = y_completo[train_index], y_completo[test_index]

                # NORMALIZAÇÃO
                scaler = StandardScaler()
                # O scaler APRENDE no treino
                X_train_cv = scaler.fit_transform(X_train_cv)
                # O scaler apenas APLICA no teste (para evitar leakage)
                X_test_cv = scaler.transform(X_test_cv)


                # Instanciar e treinar
                model = CW_RBF_KNN()
                model.train(X_train_cv, y_train_cv)

                # Avaliar o fold atual
                acc, prec, rec, f1 = model.evaluate(X_test_cv, y_test_cv, k=5)

                #adicionamos aos arrays de resultados
                fold_acc.append(acc)
                fold_prec.append(prec)
                fold_rec.append(rec)
                fold_f1.append(f1)
                fold_sel_size.append(len(model.x_train)) # Guarda quantos pontos o CNN selecionou neste fold
            
            # tamanho do treino
            orig_size = len(y_completo)
            avg_sel_size = np.mean(fold_sel_size)
            
            # Imprimir linha formatada (Os resultados dão as médias dos arrays obtidos anteriomente)
            print(f"{ficheiro[:35]:<35} | {orig_train_size:>6} | {avg_sel_size:>6} | {np.mean(fold_acc)*100:>6.1f}% | {np.mean(fold_prec):>6.2f} | {np.mean(fold_rec):>6.2f} | {np.mean(fold_f1):>6.2f}")
            
        except Exception as e:
            print(f"Erro no dataset {ficheiro}: {e}")

print("-" * len(header))
print("Avaliação empírica concluída.")

Dataset                             |   Orig |    Sel |     Acc |   Prec |    Rec |     F1
------------------------------------------------------------------------------------------
dataset_978_mfeat-factors.csv       |   1400 |  260.2 |   99.2% |   0.96 |   0.97 |   0.96
dataset_947_arsenic-male-bladder.cs |    391 |  177.0 |   97.5% |   0.93 |   0.45 |   0.59
dataset_1004_synthetic_control.csv  |    420 |   86.2 |  100.0% |   1.00 |   1.00 |   1.00
dataset_1056_mc1.csv                |   6626 |  618.0 |   99.1% |   0.40 |   0.37 |   0.37
dataset_940_water-treatment.csv     |    368 |  303.6 |   80.3% |   0.34 |   0.38 |   0.35
dataset_950_arsenic-female-lung.csv |    391 |  102.8 |   98.2% |   0.85 |   0.63 |   0.68
dataset_1014_analcatdata_dmft.csv   |    557 |  843.2 |   59.6% |   0.20 |   0.35 |   0.26
dataset_1039_hiva_agnostic.csv      |   2960 | 1161.4 |   95.7% |   0.38 |   0.29 |   0.32
dataset_1018_ipums_la_99-small.csv  |   6190 | 2567.4 |   90.0% |   0.23 |   0.24 |   0.23

# Comparação entre diferentes valores de gama (RBF)

In [7]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

class CW_RBF_KNN:
    def __init__(self):
        # Hiperparâmetro que controla a largura da "bolha" de influência do Kernel RBF
        self.gamma = None
        self.class_weights = {}
        self.feature_weights = []
        self.x_train = None
        self.y_train = None

    def load_split_train(self, X, y, test_size=0.2, random_state=42):
        """
        Recebe o dataset completo, faz o split de treino/teste, garante a estratificação 
        para dados desequilibrados e inicia o processo de Insertion Selection.
        """
        x_train_raw, x_test, y_train_raw, y_test = train_test_split(
            X, y, 
            test_size=test_size, 
            random_state=random_state, 
            stratify=y # Mantém a proporção da classe minoritária
        )
        
        # O método train aqui executa a Sequential Insertion Selection (CNN)
        self.train(x_train_raw, y_train_raw)
        
        return x_test, y_test

    def train(self, x_raw, y_raw, gamma_override=None):
        x_raw = np.array(x_raw)
        y_raw = np.array(y_raw).flatten()
        
        self.__update_internal_weights(x_raw, y_raw, gamma_override)
        
        counts = Counter(y_raw)
        minority_class = min(counts, key=counts.get)
        
        # Começa com TODOS os pontos da classe minoritária
        minority_mask = y_raw == minority_class
        self.x_train = x_raw[minority_mask]
        self.y_train = y_raw[minority_mask]
        
        # Adiciona apenas 1 ponto de cada classe maioritária para ter referência
        for label in np.unique(y_raw):
            if label != minority_class:
                idx = np.where(y_raw == label)[0][0]
                self.x_train = np.vstack([self.x_train, x_raw[idx]])
                self.y_train = np.append(self.y_train, label)
        
        # Depois aplica o sequential insertion selection apenas nos pontos maioritários        
        changed = True
        max_loops = 3
        loops = 0
        while changed and loops < max_loops:
            changed = False
            loops += 1
            for i in range(len(x_raw)):
                if y_raw[i] == minority_class:
                    continue  # já estão todos dentro
                pred = self.predict([x_raw[i]], k=5)[0]
                if pred != y_raw[i]:
                    self.x_train = np.vstack([self.x_train, x_raw[i]])
                    self.y_train = np.append(self.y_train, y_raw[i])
                    changed = True
            if changed:
                self.__update_internal_weights(self.x_train, self.y_train, gamma_override)

    def __update_internal_weights(self, x_data, y_data, gamma_override=None):
        """Função auxiliar para atualizar pesos de classes e atributos durante a inserção"""
        total_samples = len(y_data)
        counts = Counter(y_data)

        # Lógica de Gamma com Override
        if gamma_override is not None:
            self.gamma = gamma_override
        # Gamma pela mediana das distâncias (muito mais estável)
        elif len(x_data) > 1:
            # Amostra para não ser lento
            sample = x_data[:min(200, len(x_data))]
            dists = []
            for i in range(len(sample)):
                for j in range(i+1, len(sample)):
                    dists.append(np.sum(np.square(sample[i] - sample[j])))
            median_dist = np.median(dists)
            self.gamma = 1.0 / (2 * median_dist) if median_dist > 0 else 1.0
        else:
            self.gamma = 1.0 / x_data.shape[1]

        # PESOS DAS CLASSES (Suavizados com Raiz Quadrada)
        # Dá mais poder de voto à classe minoritária
        #alpha = 0.3 #na eventualidade de estarmos a dar demasiado peso à classe minoritária
        self.class_weights = {label: np.sqrt(total_samples / count) for label, count in counts.items()}

        # PESOS DAS VARIÁVEIS (Correlação Absoluta)
        if len(counts) > 1:
            unique_labels = list(counts.keys())
            y_numeric = np.where(y_data == unique_labels[0], 0, 1)
            
            f_weights = []
            for i in range(x_data.shape[1]):
                coluna = x_data[:, i]
                if np.std(coluna) == 0:
                    f_weights.append(0.0)
                else:
                    corr = np.corrcoef(coluna, y_numeric)[0, 1]
                    f_weights.append(np.abs(corr) if not np.isnan(corr) else 0.0)
            self.feature_weights = np.array(f_weights)
        else:
            self.feature_weights = np.ones(x_data.shape[1])

    def predict(self, x_data, k=5):
        x_data = np.array(x_data)
        predicts = []
        for x in x_data:
            # Calcular a Similaridade RBF em relação a todos os pontos de treino selecionados
            similarities = self.__rbf_similarity(x)
            # Ordenar de forma DECRESCENTE (no RBF, maior = melhor/mais perto)
            sorted_similarities = sorted(similarities, key=lambda t: t[0], reverse=True)
            k_nearest = sorted_similarities[:k]
            predicts.append(self.__vote(k_nearest))
        return np.array(predicts)

    def __rbf_similarity(self, point_a):
        # Diferença ponderada pelos feature_weights (Correlação)
        diferenca_quadrado = np.square(self.x_train - point_a)
        distancia_ponderada = np.sum(self.feature_weights * diferenca_quadrado, axis=1)
        # Aplicar o Kernel RBF
        similarities = np.exp(-self.gamma * distancia_ponderada)
        return list(zip(similarities, self.y_train))

    def __vote(self, neighbors):
        class_scores = {}
        for similarity, label in neighbors:
            vote_power = self.class_weights.get(label, 1.0) * similarity
            class_scores[label] = class_scores.get(label, 0) + vote_power
        
        """counts = Counter(self.y_train)
        minority = min(counts, key=counts.get)
        
        # Se a classe minoritária tem pelo menos 40% do score da maioritária, ganha
        minority_score = class_scores.get(minority, 0)
        max_score = max(class_scores.values())
        
         if minority_score >= 0.4 * max_score:
            return minority """
        return max(class_scores, key=class_scores.get)

    def evaluate(self, x_test, y_test, k=5):
        """Calcula métricas focadas em desequilíbrio para a classe minoritária."""
        y_real = np.array(y_test).flatten()
        y_pred = self.predict(x_test, k=k)
        
        counts = Counter(y_real)
        minority = min(counts, key=counts.get)
        
        tp = np.sum((y_pred == minority) & (y_real == minority))
        fp = np.sum((y_pred == minority) & (y_real != minority))
        fn = np.sum((y_pred != minority) & (y_real == minority))
        
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0
        acc = np.mean(y_pred == y_real)
        
        return acc, prec, rec, f1

In [20]:
import os
from sklearn.model_selection import StratifiedKFold
import numpy as np

# Pasta com os ficheiros CSV
pasta_datasets = 'class_imbalance'

# Lista de gammas para testar. None significa "Automático"
lista_gammas = [None, 0.1, 1.0, 10.0]

# Cabeçalho da Tabela para a avaliação empírica
header = f"{'Dataset':<35} | {'Gamma':>12} | {'Orig':>6} | {'Sel':>6} | {'Acc':>7} | {'Prec':>6} | {'Rec':>6} | {'F1':>6}"
print(header)
print("-" * len(header))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) #k fold cross validation estratificado e definimos 5 folds (n_splits)
#(o estratificado garante que a proporção é mais ou menos igual tanto no teste como no treino)

for ficheiro in os.listdir(pasta_datasets):
    if ficheiro.endswith('.csv'):
        try:
            caminho_completo = os.path.join(pasta_datasets, ficheiro)
            df = pd.read_csv(caminho_completo)
            
            # Separar features (X) da target (y)
            X_df = df.iloc[:, :-1]
            y_completo = df.iloc[:, -1].values
            
            # Transformar todas as colunas de TEXTO em NÚMEROS (One-Hot Encoding)
            # O drop_first=True evita colinearidade matemática perfeita
            X_df = pd.get_dummies(X_df, drop_first=True)
            
            # Preencher possíveis valores vazios (NaN) com 0 
            X_df = X_df.fillna(0)
            
            # Converter para array do NumPy e garantir que é tudo número (float)
            X_completo = X_df.values.astype(float)


            for g_val in lista_gammas:
            
                # Calcular tamanho do treino (antes da compressão) para a tabela
                orig_train_size = int(len(y_completo) * 0.7) 

                # Listas para guardar as métricas de cada fold
                fold_acc, fold_prec, fold_rec, fold_f1, fold_sel_size = [], [], [], [], []
            
                for train_index, test_index in skf.split(X_completo, y_completo):
                    # Dividir os dados
                    X_train_cv, X_test_cv = X_completo[train_index], X_completo[test_index]
                    y_train_cv, y_test_cv = y_completo[train_index], y_completo[test_index]

                    # NORMALIZAÇÃO
                    scaler = StandardScaler()
                    # O scaler APRENDE no treino
                    X_train_cv = scaler.fit_transform(X_train_cv)
                    # O scaler apenas APLICA no teste (para evitar leakage)
                    X_test_cv = scaler.transform(X_test_cv)


                    # Instanciar e treinar
                    model = CW_RBF_KNN()
                    model.train(X_train_cv, y_train_cv, gamma_override=g_val)

                    # Avaliar o fold atual
                    acc, prec, rec, f1 = model.evaluate(X_test_cv, y_test_cv, k=5)

                    #adicionamos aos arrays de resultados
                    fold_acc.append(acc)
                    fold_prec.append(prec)
                    fold_rec.append(rec)
                    fold_f1.append(f1)
                    fold_sel_size.append(len(model.x_train)) # Guarda quantos pontos o CNN selecionou neste fold
            
                # tamanho do treino
                orig_size = len(y_completo)
                avg_sel_size = np.mean(fold_sel_size)

                # Formatação do nome do Gamma para a tabela
                g_str = f"{model.gamma:.2f}" if g_val is not None else f"Auto({model.gamma:.2f})"
            
                # Imprimir linha formatada (Os resultados dão as médias dos arrays obtidos anteriomente)
                print(f"{ficheiro[:35]:<35} | {g_str:>12} | {orig_train_size:>6} | {avg_sel_size:>6} | {np.mean(fold_acc)*100:>6.1f}% | {np.mean(fold_prec):>6.2f} | {np.mean(fold_rec):>6.2f} | {np.mean(fold_f1):>6.2f}")
            
        except Exception as e:
            print(f"Erro no dataset {ficheiro}: {e}")

print("-" * len(header))
print("Avaliação empírica concluída.")

Dataset                             |        Gamma |   Orig |    Sel |     Acc |   Prec |    Rec |     F1
---------------------------------------------------------------------------------------------------------
Erro no dataset dataset_978_mfeat-factors.csv: CW_RBF_KNN.train() got an unexpected keyword argument 'gamma_override'
Erro no dataset dataset_947_arsenic-male-bladder.csv: CW_RBF_KNN.train() got an unexpected keyword argument 'gamma_override'
Erro no dataset dataset_1004_synthetic_control.csv: CW_RBF_KNN.train() got an unexpected keyword argument 'gamma_override'
Erro no dataset dataset_1056_mc1.csv: CW_RBF_KNN.train() got an unexpected keyword argument 'gamma_override'
Erro no dataset dataset_940_water-treatment.csv: CW_RBF_KNN.train() got an unexpected keyword argument 'gamma_override'
Erro no dataset dataset_950_arsenic-female-lung.csv: CW_RBF_KNN.train() got an unexpected keyword argument 'gamma_override'
Erro no dataset dataset_1014_analcatdata_dmft.csv: CW_RBF_KNN.train()